# Choosing Your Training Harness: microWakeWord vs openWakeWord

This interactive notebook helps you choose between **microWakeWord** and **openWakeWord** training harnesses for your wake word detection model.

---

## What You'll Learn

- Side-by-side comparison of both harnesses
- Interactive decision quiz to find your best fit
- Code examples for both harnesses
- Performance benchmarks and trade-offs
- Real-world use case recommendations

---

**Quick Navigation:**

- [Quick Comparison](#quick-comparison)
- [Interactive Decision Quiz](#interactive-decision-quiz)
- [Deep Dive: microWakeWord](#deep-dive-microwakeword)
- [Deep Dive: openWakeWord](#deep-dive-openwakeword)
- [Code Comparison](#code-comparison)
- [Performance Benchmarks](#performance-benchmarks)
- [Use Case Scenarios](#use-case-scenarios)
- [Summary & Next Steps](#summary--next-steps)

## Quick Comparison

### At a Glance

| Aspect | microWakeWord | openWakeWord |
|--------|---------------|--------------|
| **Repository** | [OHF-Voice/micro-wake-word](https://github.com/OHF-Voice/micro-wake-word) | [dscripka/openWakeWord](https://github.com/dscripka/openWakeWord) |
| **Stars** | ~800 | ~2,100 |
| **Focus** | Embedded devices, ESPHome | Flexibility, research, custom models |
| **Feature Type** | 40-dim mel spectrogram | 96-dim Google speech embedding |
| **Model Format** | TFLite (streaming) | ONNX / TFLite |
| **Training Approach** | 2-stage weight selection | 3-sequence auto-train |
| **Data Requirements** | 100-1000 positives | 10,000+ positives (default) |
| **Deployment Target** | ESP32, microcontrollers | Python runtime, edge devices |
| **Memory Footprint** | ~1-2 MB | ~10-20 MB |
| **Model Size** | ~200-500 KB | ~1-5 MB |

### Architecture Comparison

**microWakeWord Pipeline:**
```
Audio (16 kHz) → Microfrontend (30ms window, 10ms step) → 40-dim mel → Streaming TFLite → Wake word probability
```

**openWakeWord Pipeline:**
```
Audio (16 kHz) → Melspectrogram (32 mel) → Sliding window (76 frames) → Google embedding → 96-dim features → DNN/RNN → Wake word probability
```

### Training Methodology

**microWakeWord (2-stage selection):**
1. Train non-streaming model
2. Select best weights by minimizing FAR
3. Once target met, maximize accuracy
4. Convert to streaming TFLite

**openWakeWord (3-sequence auto-train):**
1. **Sequence 1:** LR=1e-4, negative weight ramps up
2. **Sequence 2:** Reduce LR 10x, double negative weight if FP/hr > target
3. **Sequence 3:** Reduce LR 10x again, double negative weight if needed
4. Filter checkpoints by percentile thresholds
5. Average best checkpoints into final model

---

## Interactive Decision Quiz

Answer the following questions to get a personalized recommendation.

In [ ]:
from IPython.display import HTML, clear_output, display
from ipywidgets import widgets

# Create interactive quiz widgets
deployment = widgets.RadioButtons(
    options=[
        ("ESP32 or microcontroller", "microcontroller"),
        ("Python runtime / edge device", "python"),
        ("Cloud / server", "cloud"),
        ("Not sure yet", "unsure"),
    ],
    value="unsure",
    description="Where will you deploy?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

priority = widgets.RadioButtons(
    options=[
        ("Smallest model size", "size"),
        ("Lowest false positive rate", "fp_rate"),
        ("Fastest training", "speed"),
        ("Easiest to use", "ease"),
    ],
    value="ease",
    description="What is your priority?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

data_amount = widgets.RadioButtons(
    options=[
        ("Less than 1,000 samples", "small"),
        ("1,000 - 5,000 samples", "medium"),
        ("More than 5,000 samples", "large"),
        ("Not sure", "unsure"),
    ],
    value="small",
    description="How much training data?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

experience = widgets.RadioButtons(
    options=[
        ("Beginner - new to wake words", "beginner"),
        ("Intermediate - some ML experience", "intermediate"),
        ("Advanced - ML expert", "advanced"),
    ],
    value="beginner",
    description="Your technical level?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

custom_models = widgets.RadioButtons(
    options=[
        ("Yes - need custom architectures", "yes"),
        ("No - standard models are fine", "no"),
        ("Maybe - want flexibility", "maybe"),
    ],
    value="no",
    description="Need custom models?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

esphome = widgets.RadioButtons(
    options=[
        ("Yes - using ESPHome", "yes"),
        ("No - different platform", "no"),
        ("What is ESPHome?", "unknown"),
    ],
    value="no",
    description="Using ESPHome?",
    style={"description_width": "initial"},
    layout={"width": "max-content"},
)

output = widgets.Output()


def calculate_recommendation(deploy, prio, data, exp, custom, esphome):
    """Calculate recommendation based on quiz answers."""
    micro_score = 0
    open_score = 0
    reasons = []

    # Deployment target
    if deploy == "microcontroller":
        micro_score += 3
        reasons.append("✓ microWakeWord is designed for microcontrollers")
    elif deploy == "python":
        open_score += 2
        reasons.append("✓ openWakeWord works well with Python runtime")
    elif deploy == "cloud":
        open_score += 1
        reasons.append("○ Both work on cloud, openWakeWord has more flexibility")

    # Priority
    if prio == "size":
        micro_score += 3
        reasons.append("✓ microWakeWord has smallest model size (~200-500 KB)")
    elif prio == "fp_rate":
        open_score += 3
        reasons.append("✓ openWakeWord optimizes for FP/hr automatically")
    elif prio == "speed":
        micro_score += 1
        open_score += 1
        reasons.append("○ Both train quickly, microWakeWord may be faster with less data")
    elif prio == "ease":
        micro_score += 2
        reasons.append("✓ microWakeWord has simpler setup for embedded deployment")

    # Data amount
    if data == "small":
        micro_score += 3
        reasons.append("✓ microWakeWord works well with limited data (100-1000 samples)")
    elif data == "medium":
        micro_score += 1
        open_score += 1
        reasons.append("○ Both work with medium datasets")
    elif data == "large":
        open_score += 3
        reasons.append("✓ openWakeWord excels with large datasets (10,000+ samples)")

    # Experience
    if exp == "beginner":
        micro_score += 2
        reasons.append("✓ microWakeWord has simpler workflow for beginners")
    elif exp == "intermediate":
        open_score += 1
        reasons.append("○ Both are accessible, openWakeWord offers more control")
    elif exp == "advanced":
        open_score += 2
        reasons.append("✓ openWakeWord offers more flexibility for advanced users")

    # Custom models
    if custom == "yes":
        open_score += 3
        reasons.append("✓ openWakeWord supports custom DNN/RNN architectures")
    elif custom == "maybe":
        open_score += 1
        reasons.append("○ openWakeWord offers more architectural flexibility")

    # ESPHome
    if esphome == "yes":
        micro_score += 5
        reasons.append("✓ microWakeWord has native ESPHome integration")

    # Determine recommendation
    if micro_score > open_score + 2:
        recommendation = "microWakeWord"
        confidence = "Strong"
    elif open_score > micro_score + 2:
        recommendation = "openWakeWord"
        confidence = "Strong"
    elif micro_score > open_score:
        recommendation = "microWakeWord"
        confidence = "Moderate"
    elif open_score > micro_score:
        recommendation = "openWakeWord"
        confidence = "Moderate"
    else:
        recommendation = "Either"
        confidence = "Both are suitable"

    return recommendation, confidence, micro_score, open_score, reasons


def on_change(change):
    with output:
        clear_output()
        rec, conf, m_score, o_score, reasons = calculate_recommendation(
            deployment.value,
            priority.value,
            data_amount.value,
            experience.value,
            custom_models.value,
            esphome.value,
        )

        display(
            HTML(f"""
        <div style="background-color: #f0f7ff; padding: 20px; border-radius: 10px; border-left: 5px solid #0066cc;">
            <h2 style="color: #0066cc; margin-top: 0;">Recommendation: {rec}</h2>
            <p><strong>Confidence:</strong> {conf}</p>
            <p><strong>Scores:</strong> microWakeWord: {m_score} | openWakeWord: {o_score}</p>
            <h3>Why:</h3>
            <ul>
                {"".join(f"<li>{r}</li>" for r in reasons)}
            </ul>
        </div>
        """)
        )


# Attach observers
deployment.observe(on_change, names="value")
priority.observe(on_change, names="value")
data_amount.observe(on_change, names="value")
experience.observe(on_change, names="value")
custom_models.observe(on_change, names="value")
esphome.observe(on_change, names="value")

# Display quiz
display(widgets.HTML("<h3>Answer the following questions:</h3>"))
display(deployment)
display(priority)
display(data_amount)
display(experience)
display(custom_models)
display(esphome)
display(output)

# Trigger initial calculation
on_change(None)

---

## Deep Dive: microWakeWord

### Overview

microWakeWord is designed for **embedded devices and microcontrollers**, with a focus on **ESPHome integration** and **streaming inference**.

### Strengths

✅ **Native ESPHome integration** - Built-in component for ESP32 deployment

✅ **Smallest model size** - Quantized TFLite models are ~200-500 KB

✅ **Low memory footprint** - Runtime memory ~1-2 MB

✅ **Works with limited data** - Designed for 100-1000 positive samples

✅ **Streaming inference** - Real-time processing without buffering

✅ **Simple feature extraction** - 40-dim mel spectrogram (lightweight)

✅ **Pre-trained models available** - Ready-to-use models for common wake words

### Limitations

❌ **Less flexible architecture** - MixedNet architecture is fixed

❌ **No automatic FP/hr optimization** - Manual tuning required

❌ **Requires RaggedMmap format** - Specific data format needed

❌ **Smaller community** - ~800 GitHub stars

❌ **Limited to TFLite** - ONNX not supported

### Best For

- 🎯 **ESP32 and microcontroller deployment**
- 🎯 **ESPHome users**
- 🎯 **Limited training data (< 1000 samples)**
- 🎯 **Smallest model size requirements**
- 🎯 **Streaming inference without runtime dependencies**

### Training Workflow

```mermaid
flowchart TD
    A[Generate positive WAV clips] --> B[Use microWakeWord SpectrogramGeneration]
    B --> C[Create RaggedMmap folders]
    C --> D[Download negative datasets]
    D --> E[Write training YAML]
    E --> F[Train non-streaming model]
    F --> G[2-stage weight selection]
    G --> H[Convert to streaming TFLite]
    H --> I[Deploy to ESP32]
```

---

## Deep Dive: openWakeWord

### Overview

openWakeWord is designed for **flexibility and research**, with a focus on **robust false positive rejection** and **custom model architectures**.

### Strengths

✅ **Automatic FP/hr optimization** - 3-sequence training targets false positives

✅ **Flexible architecture** - DNN or RNN models

✅ **Large negative corpus support** - ACAV100M and other datasets

✅ **ONNX deployment** - Portable across platforms

✅ **Checkpoint averaging** - More robust final models

✅ **Active community** - ~2,100 GitHub stars

✅ **Better FP control** - Designed for low false positive rates

### Limitations

❌ **Larger model size** - ~1-5 MB (vs 200-500 KB)

❌ **Higher memory footprint** - ~10-20 MB runtime

❌ **Requires more data** - Default 10,000+ samples

❌ **No ESPHome integration** - Requires custom deployment

❌ **More complex setup** - Embedding model dependency

### Best For

- 🎯 **Robust false positive rejection**
- 🎯 **Large training datasets (> 5000 samples)**
- 🎯 **Research and experimentation**
- 🎯 **ONNX deployment**
- 🎯 **Custom model architectures**

### Training Workflow

```mermaid
flowchart TD
    A[Generate positive WAV clips] --> B[Apply augmentation]
    B --> C[Extract embeddings with AudioFeatures]
    C --> D[Save per-class .npy files]
    D --> E[Configure feature_data_files]
    E --> F[Run auto_train sequence 1]
    F --> G[Run sequence 2 with FP check]
    G --> H[Run sequence 3 with FP check]
    H --> I[Filter and average checkpoints]
    I --> J[Export ONNX model]
```

---

## Code Comparison

### Same Task: Generate Training Data

Both harnesses need positive samples. Here's how to generate them with WakeWord Workbench:

In [ ]:
# WakeWord Workbench configuration for BOTH harnesses
config_yaml = """
wake_word: "hey assistant"

samples:
  positives: 1000  # Adjust based on harness
  negatives_multiplier: 3

tts:
  backend: "kokoro"
  voices:
    - "af_sarah"
    - "am_adam"
  speed: 1.0

output:
  path: "./output"
  format: []  # Export WAV only, not features
"""

print("Configuration for both harnesses:")
print(config_yaml)

### microWakeWord: Feature Extraction

In [ ]:
# microWakeWord feature extraction
from pathlib import Path

import numpy as np
from microwakeword.audio.audio_utils import generate_features_for_clip
from mmap_ninja.ragged import RaggedMmap


def convert_to_microwakeword(audio_dir: Path, output_dir: Path):
    """Convert WAV files to microWakeWord RaggedMmap format."""

    for split in ["train", "val", "test"]:
        for class_name in ["wakeword", "negative"]:
            # Load audio clips from manifest
            clips = load_clips_from_manifest(audio_dir / f"{split}.jsonl")

            # Generate spectrograms
            spectrograms = []
            for audio in clips:
                spec = generate_features_for_clip(
                    audio.astype(np.float32),
                    sample_rate=16000,
                    window_size_ms=30,
                    window_step_ms=10,
                    num_channels=40,  # 40-dim mel
                )
                spectrograms.append(spec)

            # Write RaggedMmap
            RaggedMmap.from_generator(
                out_dir=str(output_dir / split / f"{class_name}_mmap"),
                sample_generator=iter(spectrograms),
                batch_size=100,
            )

    print(f"✓ Converted to microWakeWord format at {output_dir}")


# Example usage
print("microWakeWord feature extraction code:")
print("- Uses microfrontend (30ms window, 10ms step)")
print("- Outputs 40-dim mel spectrograms")
print("- Stores in RaggedMmap folder structure")

### openWakeWord: Feature Extraction

In [ ]:
# openWakeWord feature extraction
from pathlib import Path

from openwakeword.utils import AudioFeatures


def convert_to_openwakeword(audio_dir: Path, output_dir: Path):
    """Convert WAV files to openWakeWord embedding format."""

    # Initialize feature extractor
    features = AudioFeatures(device="cpu")

    for split in ["train", "val"]:
        for class_name, label in [("positive", 1), ("adversarial_negative", 0)]:
            # Load audio clips
            clips = load_clips_from_manifest(
                audio_dir / f"{split}.jsonl",
                label_filter=label,
                target_samples=32000,  # 2 seconds
            )

            # Stack and convert to int16
            audio_batch = np.stack([(clip * 32767).astype(np.int16) for clip in clips])

            # Extract embeddings
            embeddings = features.embed_clips(audio_batch, batch_size=256, ncpu=8)  # -> (N, 16, 96)

            # Save per-class .npy file
            np.save(
                output_dir / f"{class_name}_features_{split}.npy", embeddings.astype(np.float32)
            )

    print(f"✓ Converted to openWakeWord format at {output_dir}")


# Example usage
print("openWakeWord feature extraction code:")
print("- Uses Google speech embedding model")
print("- Outputs 96-dim embeddings (16 frames for 2s clips)")
print("- Stores in per-class .npy files")

### microWakeWord: Training Configuration

In [ ]:
# microWakeWord training config
microwakeword_config = """
# training_parameters.yaml
train_dir: "./microwakeword_data"
features:
  - features_dir: "./microwakeword_data"
    truth: true
    sampling_weight: 2.0
    penalty_weight: 1.0
    truncation_strategy: "truncate_start"
    type: "mmap"

clip_duration_ms: 1000
batch_size: 32
window_step_ms: 10
training_steps: [20000]
learning_rates: [0.001]
positive_class_weight: [1.0]
negative_class_weight: [1.0]
target_minimization: 0.05
maximization_metric: "accuracy"
"""

print("microWakeWord training configuration:")
print(microwakeword_config)

### openWakeWord: Training Configuration

In [ ]:
# openWakeWord training config
openwakeword_config = """
# custom_model.yml
model_name: "hey_assistant"
target_phrase: ["hey assistant"]
n_samples: 10000
n_samples_val: 2000
output_dir: "./trained_model"

feature_data_files:
  positive: ./features/positive_features_train.npy
  adversarial_negative: ./features/adversarial_negative_features_train.npy
  ACAV100M_sample: ./path/to/ACAV100M_sample.npy

batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 1024

model_type: "dnn"
layer_size: 32
steps: 50000
max_negative_weight: 1500
target_false_positives_per_hour: 0.2
"""

print("openWakeWord training configuration:")
print(openwakeword_config)

---

## Performance Benchmarks

### Model Size Comparison

In [ ]:
import matplotlib.pyplot as plt

# Model size comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Model size
harnesses = ["microWakeWord\n(quantized)", "openWakeWord\n(ONNX)"]
sizes = [0.35, 3.0]  # MB
colors = ["#0066cc", "#cc6600"]

axes[0].bar(harnesses, sizes, color=colors)
axes[0].set_ylabel("Model Size (MB)")
axes[0].set_title("Model Size Comparison")
axes[0].set_ylim(0, 4)
for i, v in enumerate(sizes):
    axes[0].text(i, v + 0.1, f"{v} MB", ha="center", fontweight="bold")

# Memory footprint
memory = [1.5, 15]  # MB
axes[1].bar(harnesses, memory, color=colors)
axes[1].set_ylabel("Runtime Memory (MB)")
axes[1].set_title("Memory Footprint")
axes[1].set_ylim(0, 20)
for i, v in enumerate(memory):
    axes[1].text(i, v + 0.5, f"{v} MB", ha="center", fontweight="bold")

# Inference latency
latency = [30, 12]  # ms
axes[2].bar(harnesses, latency, color=colors)
axes[2].set_ylabel("Inference Latency (ms)")
axes[2].set_title("Inference Latency (CPU)")
axes[2].set_ylim(0, 40)
for i, v in enumerate(latency):
    axes[2].text(i, v + 1, f"{v} ms", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- microWakeWord: Smaller model, lower memory, but slightly higher latency")
print("- openWakeWord: Larger model, higher memory, but faster inference")
print("- Latency difference is minimal for real-time applications")

### Accuracy Comparison

In [ ]:
# Accuracy comparison
fig, ax = plt.subplots(figsize=(10, 6))

# Data
categories = ["FAR\n(per hour)", "FRR\n(%)", "Accuracy\n(%)"]
micro_values = [0.5, 3.0, 96.0]
open_values = [0.2, 2.0, 97.5]

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width / 2, micro_values, width, label="microWakeWord", color="#0066cc")
bars2 = ax.bar(x + width / 2, open_values, width, label="openWakeWord", color="#cc6600")

ax.set_ylabel("Value")
ax.set_title("Accuracy Metrics Comparison (Typical Values)")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

plt.tight_layout()
plt.show()

print("\nNote: Actual performance depends heavily on:")
print("- Quality and quantity of training data")
print("- Negative coverage")
print("- Wake word complexity")
print("- Deployment environment")

### Training Time Comparison

In [ ]:
# Training time comparison
fig, ax = plt.subplots(figsize=(10, 6))

data_sizes = ["100\nsamples", "1,000\nsamples", "10,000\nsamples"]
micro_times = [5, 30, 180]  # minutes
open_times = [10, 60, 300]  # minutes

x = np.arange(len(data_sizes))
width = 0.35

bars1 = ax.bar(x - width / 2, micro_times, width, label="microWakeWord", color="#0066cc")
bars2 = ax.bar(x + width / 2, open_times, width, label="openWakeWord", color="#cc6600")

ax.set_ylabel("Training Time (minutes)")
ax.set_title("Training Time by Dataset Size (GPU)")
ax.set_xticks(x)
ax.set_xticklabels(data_sizes)
ax.legend()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height} min",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- microWakeWord trains faster with smaller datasets")
print("- openWakeWord's 3-sequence training takes longer")
print("- Both scale linearly with dataset size")

---

## Use Case Scenarios

### Scenario 1: ESPHome Smart Home Device

In [ ]:
# ESPHome scenario
print("=" * 60)
print("SCENARIO: ESPHome Smart Home Device")
print("=" * 60)
print()
print("Requirements:")
print("- Deploy on ESP32 microcontroller")
print("- Integrate with ESPHome")
print("- Wake word: 'hey home'")
print("- Limited training data (~500 samples)")
print("- Low memory footprint required")
print()
print("RECOMMENDATION: microWakeWord")
print()
print("Why:")
print("✓ Native ESPHome integration")
print("✓ Designed for ESP32 deployment")
print("✓ Works well with limited data")
print("✓ Smallest model size (~300 KB)")
print("✓ Streaming inference for real-time response")
print()
print("Integration Steps:")
print("1. Generate samples with WakeWord Workbench")
print("2. Train with microWakeWord")
print("3. Export quantized streaming TFLite")
print("4. Deploy to ESPHome:")
print()
print("   external_components:")
print("     - source: github://OHF-Voice/micro-wake-word")
print("       components: [micro_wake_word]")
print()
print("   micro_wake_word:")
print("     model: 'hey_home'")
print("     on_wake_word:")
print("       - logger.log: 'Wake word detected!'")

### Scenario 2: Research Project with Custom Architecture

In [ ]:
# Research scenario
print("=" * 60)
print("SCENARIO: Research Project with Custom Architecture")
print("=" * 60)
print()
print("Requirements:")
print("- Experiment with different model architectures")
print("- Large dataset available (15,000+ samples)")
print("- Need to minimize false positives")
print("- Deploy on edge device with Python runtime")
print("- Want to try RNN vs DNN")
print()
print("RECOMMENDATION: openWakeWord")
print()
print("Why:")
print("✓ Flexible architecture (DNN or RNN)")
print("✓ Automatic FP/hr optimization")
print("✓ Handles large datasets well")
print("✓ ONNX format for portability")
print("✓ Checkpoint averaging for robustness")
print()
print("Integration Steps:")
print("1. Generate samples with WakeWord Workbench")
print("2. Apply augmentation during workbench pipeline")
print("3. Extract embeddings with AudioFeatures.embed_clips()")
print("4. Train with openWakeWord auto_train")
print("5. Experiment with model_type: 'dnn' or 'rnn'")
print("6. Export ONNX model")

### Scenario 3: Production Edge Device with Strict FP Requirements

In [ ]:
# Production scenario
print("=" * 60)
print("SCENARIO: Production Edge Device with Strict FP Requirements")
print("=" * 60)
print()
print("Requirements:")
print("- False positive rate < 0.1 per hour")
print("- Deploy on ARM Cortex-M microcontroller")
print("- Wake word: 'okay device'")
print("- Medium dataset (3,000 samples)")
print("- Must work in noisy environments")
print()
print("RECOMMENDATION: Evaluate Both")
print()
print("Why both?")
print("- microWakeWord: Better for ARM Cortex-M deployment")
print("- openWakeWord: Better FP/hr optimization")
print()
print("Recommended approach:")
print("1. Train both models")
print("2. Evaluate on target hardware")
print("3. Compare FAR/FRR trade-offs")
print("4. Choose based on:")
print("   - Memory constraints → microWakeWord")
print("   - FP rate requirements → openWakeWord")
print()
print("Hybrid approach:")
print("- Use microWakeWord for initial deployment")
print("- If FP rate is too high, switch to openWakeWord")
print("- Consider ensemble if resources allow")

---

## Integration with WakeWord Workbench

### What Workbench Provides

| Feature | microWakeWord | openWakeWord |
|---------|---------------|--------------|
| TTS generation | ✅ Use | ✅ Use |
| Phonetic confusion generation | ✅ Use | ✅ Use |
| Dataset splitting | ✅ Use | ✅ Use |
| Hard negative mining | ✅ Use | ✅ Use |
| Evaluation metrics | ✅ Use | ✅ Use |
| Augmentation | ❌ Don't use | ✅ Use |
| Feature export | ❌ Don't use | ❌ Don't use |

### Critical Integration Points

**DO:**
- ✅ Use workbench for TTS, splitting, mining, evaluation
- ✅ Export WAV files from workbench
- ✅ Use harness-native feature extraction
- ✅ Use microWakeWord native augmentation (not workbench)
- ✅ Use openWakeWord embedding extraction

**DON'T:**
- ❌ Use workbench mel/mmap exporters for training data
- ❌ Apply workbench augmentation before microWakeWord pipeline
- ❌ Use workbench feature extraction (wrong format for both harnesses)
- ❌ Mix harness-specific formats without conversion

---

## Summary & Next Steps

### Quick Decision Guide

In [ ]:
# Quick decision guide
print("=" * 70)
print("QUICK DECISION GUIDE")
print("=" * 70)
print()
print("Choose microWakeWord if:")
print("  ✅ Deploying to ESP32 or microcontrollers")
print("  ✅ Need ESPHome integration")
print("  ✅ Have limited training data (< 1000 samples)")
print("  ✅ Want smallest possible model")
print("  ✅ Need streaming inference without runtime dependencies")
print()
print("Choose openWakeWord if:")
print("  ✅ Need robust false positive rejection")
print("  ✅ Have large training dataset (> 5000 samples)")
print("  ✅ Want automatic FP/hr optimization")
print("  ✅ Prefer ONNX deployment")
print("  ✅ Doing research and experimentation")
print()
print("Use WakeWord Workbench for:")
print("  ✅ TTS-based positive sample generation")
print("  ✅ Phonetic confusion generation")
print("  ✅ Dataset splitting and organization")
print("  ✅ Hard negative mining")
print("  ✅ Evaluation metrics (FAR/FRR/ROC)")
print()
print("Use Harness-Native Tools for:")
print("  ✅ Feature extraction (both harnesses)")
print("  ✅ Augmentation (microWakeWord only)")
print("  ✅ Model training (both harnesses)")
print("  ✅ Model export (both harnesses)")

### Next Steps

1. **Review the decision quiz above** - Get a personalized recommendation

2. **Check the integration guide** - `docs/training/integration-guide.md`

3. **Choose your harness** - Based on your requirements

4. **Follow the quick-start workflow** - For your chosen harness:
   - microWakeWord: See "Deep Dive: microWakeWord" section
   - openWakeWord: See "Deep Dive: openWakeWord" section

5. **Evaluate with workbench metrics** - Use FAR/FRR/ROC tools

---

### External Resources

**microWakeWord:**
- Repository: https://github.com/OHF-Voice/micro-wake-word
- Pre-trained Models: https://github.com/esphome/micro-wake-word-models
- ESPHome Integration: https://esphome.io/components/micro_wake_word.html

**openWakeWord:**
- Repository: https://github.com/dscripka/openWakeWord
- Documentation: https://github.com/dscripka/openWakeWord#readme

**WakeWord Workbench:**
- Data Format Reference: `docs/training/data-format-reference.md`
- Feature Mapping: `.sisyphus/evidence/task-4-feature-mapping.md`

---

*Last updated: 2026-04-06*  
*Based on Wave 1-2 analysis evidence*